In [ ]:
import os
os.chdir("C:/Users/Administrator/PythonProjects/discharge_queich/src")

from pathlib import Path
import yaml
from pydantic_settings import BaseSettings
from discharge_queich.configs import settings

In [21]:
from pathlib import Path
from pydantic import BaseModel
from pydantic_settings import SettingsConfigDict

from discharge_queich.configs.database import DatabaseSettings
from discharge_queich.configs.dashboard import DashboardSettings
from discharge_queich.configs.ingestion import IngestionSettings
from discharge_queich.configs.model import ModelSettings
from discharge_queich.configs.scheduler import SchedulerSettings


class Settings(BaseModel):
    
    model_config = SettingsConfigDict(
        env_nested_delimiter="__",
        # yaml_file="/discharge_queich/configs/yaml/dashboard.yaml",   # assuming you load yaml
    )
    
    database: DatabaseSettings
    dashboard: DashboardSettings
    ingestion: IngestionSettings
    model: ModelSettings
    scheduler: SchedulerSettings

In [41]:
# CONFIG_DIR = Path(__file__).parent / "yaml"
CONFIG_DIR = Path("discharge_queich/configs/") / "yaml"

def load_yaml(filename: str) -> dict:
    with open(CONFIG_DIR / filename, "r") as f:
        return yaml.safe_load(f)

In [161]:
def apply_env_override(config: dict) -> dict:
    
    for key, val in get_environ().items():
        if "API_URL" not in key:
            continue
        
        path = key.lower().split("__")
        
        current = config
        
        for section in path[:-1]:
            
            if section not in current:
                break
            
            current = current[section]
        
        else:
            current[path[-1]] = val
            
    return config

In [162]:
def load_settings() -> Settings:

    config = {
        "database": load_yaml("database.yaml"),
        "dashboard": load_yaml("dashboard.yaml"),
        "ingestion": load_yaml("ingestion.yaml"),
        "model": load_yaml("model.yaml"),
        "scheduler": load_yaml("scheduler.yaml"),
    }
    
    config = apply_env_override(config=config)
    
    return Settings.model_validate(config)

In [165]:
def get_environ():
    config= {
        'HOSTNAME': '393af5dc5797',
        'HOME': '/root',
        'GPG_KEY': 'A035C8C19219BA821ECEA86B64E628F8D684696D',
        'PYTHON_SHA256': 'de6517421601e39a9a3bc3e1bc4c7b2f239297423ee05e282598c83ec0647505',
        'TERM': 'xterm',
        'PATH': '/app/.venv/bin:/usr/local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin',
        'LANG': 'C.UTF-8',
        # 'DASHBOARD__API_URL': 'http://api:8000/forecast',
        'PYTHON_VERSION': '3.10.20',
        'PWD': '/app'
        }
    
    return config

In [166]:
config = load_settings() 
config.dashboard.api_url

'http://localhost:8000/forecast'

In [167]:
from pydantic_settings import YamlConfigSettingsSource, EnvSettingsSource

In [ ]:
YamlConfigSettingsSource